## Compute Bias-related Indicators

**Indicator 1** - Sentiment Polarity and Score - VADER (works very well for headlines)

- `sentiment_polarity`: positive / negative / neutral
- `sentiment_score`: compound score (-1 to 1)

**Indicator 2** - Emotion Intensity (strength of emotion, regardless of positive or negative)

- `emotional_intensity = |sentimental_score|`

**Indicator 3** - Subjectivity (this is HUGE for bias detection)

- `0.0 -> factual and 1.0 -> opinionated` 

**Indicator 4** - Loaded / Framing Language (detect emotionally or rhetorically loaded words)

```py
FRAMING_WORDS = [
    "shocking","disastrous","failure","crisis","outrage",
    "slams","blasts","hits back","reveals","quips",
    "furious","reckless","toxic","controversial",
    "exposes","faces backlash","sparks outrage"
]
```

In [1]:
import pandas as pd
import re
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

In [2]:
df = pd.read_csv("headlines_standardized.csv")
df.head()

,id,headline,country,label,headline_clean
0,1,0% salary hike even after great performance? E...,India,Sensational/Clickbait,0 salary hike even after great performance emp...
1,2,Learjet 45: The jet Ajit Pawar took for his fi...,India,Human-Interest,learjet 45 the jet ajit pawar took for his fin...
2,3,Lead the next wave of consumer businesses with...,India,Promotional,lead the next wave of consumer businesses with...
3,4,CUET UG 2026 registration ends in two days: Ho...,India,Neutral/Factual,cuet ug 2026 registration ends in two days how...
4,5,Employee who was denied WFH by boss due to los...,India,Human-interest,employee who was denied wfh by boss due to los...


In [3]:
analyzer = SentimentIntensityAnalyzer()

FRAMING_WORDS = [
    "shocking","disastrous","failure","crisis","outrage",
    "slams","blasts","reveals","quips","furious",
    "reckless","toxic","controversial","exposes",
    "backlash","sparks"
]

In [4]:
def sentiment_features(text):
    scores = analyzer.polarity_scores(text)
    compound = scores["compound"]

    if compound >= 0.05:
        polarity = "positive"
    elif compound <= -0.05:
        polarity = "negative"
    else:
        polarity = "neutral"

    return polarity, compound, abs(compound)

def subjectivity_score(text):
    return TextBlob(text).sentiment.subjectivity

def framing_score(text):
    text = text.lower()
    hits = sum(1 for w in FRAMING_WORDS if re.search(rf"\b{w}\b", text))
    return hits

In [5]:
df[["sentiment_polarity",
    "sentiment_score",
    "emotional_intensity"]] = df["headline_clean"].apply(
        lambda x: pd.Series(sentiment_features(x))
)

df["subjectivity"] = df["headline_clean"].apply(subjectivity_score)
df["framing_score"] = df["headline_clean"].apply(framing_score)

df["loaded_language"] = df["framing_score"].apply(
    lambda x: "yes" if x > 0 else "no"
)

df.head()

,id,headline,country,label,headline_clean,sentiment_polarity,sentiment_score,emotional_intensity,subjectivity,framing_score,loaded_language
0,1,0% salary hike even after great performance? E...,India,Sensational/Clickbait,0 salary hike even after great performance emp...,positive,0.6249,0.6249,0.7500,1,yes
1,2,Learjet 45: The jet Ajit Pawar took for his fi...,India,Human-Interest,learjet 45 the jet ajit pawar took for his fin...,neutral,0.0000,0.0000,1.0000,0,no
2,3,Lead the next wave of consumer businesses with...,India,Promotional,lead the next wave of consumer businesses with...,neutral,0.0000,0.0000,0.0000,0,no
3,4,CUET UG 2026 registration ends in two days: Ho...,India,Neutral/Factual,cuet ug 2026 registration ends in two days how...,neutral,0.0000,0.0000,0.0000,0,no
4,5,Employee who was denied WFH by boss due to los...,India,Human-interest,employee who was denied wfh by boss due to los...,negative,-0.3818,0.3818,0.6875,1,yes


In [6]:
df.to_csv("headlines_annotated.csv", index=False)

print("✅ Bias indicators computed and saved.")

✅ Bias indicators computed and saved.
